# FP_NB_DataQuality
**Northbrook Lending: automated data quality checks across Bronze, Silver and Gold**

Every run appends one row per check to **`FP_Gold.dbo.DQ_Results`**, so DQ history can be trended in Power BI (the *Data Quality* report page).

| Check type | Examples |
|---|---|
| **Row count reconciliation** | Bronze (mirror) vs Silver for every table; current customers vs source |
| **Completeness (null checks)** | keys and mandatory attributes |
| **Uniqueness** | primary keys; exactly one current SCD2 row per customer |
| **Referential integrity** | Loan → Customer/Product/Branch, Repayment → Loan; Gold facts → dimensions |
| **Validity / value ranges** | credit score 0–999, age 18–100, interest rate 0–30%, paid ≤ due, UK postcode and e-mail formats, dates not in the future |
| **Reasonableness** | portfolio arrears rate within an expected band (warning only) |

**Severity:** `Critical` failures make the notebook raise an error, which triggers the pipeline's failure path. `Warning` failures are recorded but don't stop the load.


In [ ]:
# Parameters - the pipeline overrides these at run time; run_id stays empty when you run the notebook by hand
run_id = ""
as_at_date = "2026-11-19"   # reporting as-at date
fail_on_critical = True   # True = stop the pipeline when a Critical check fails

In [ ]:
# ---------- Setup ----------
# uuid for run IDs, json for the exit value, datetime for dates and timestamps
import uuid, json, datetime as dt
# Spark SQL functions such as col, sum and isNull - imported as F
from pyspark.sql import functions as F
# Spark data types - used to define the layout of the DQ results table
from pyspark.sql.types import (StructType, StructField, StringType, LongType, DoubleType, TimestampType)

# Use the pipeline run ID if one was passed in, otherwise make a random one for manual runs
run_id = run_id or str(uuid.uuid4())
# Time this run started, in UTC - stamped on every result row
run_ts = dt.datetime.utcnow()
# Turn the as-at date text into a real date
AS_AT = dt.date.fromisoformat(as_at_date)

# Find the OneLake root folder of a lakehouse by name, in whichever workspace this runs
def lakehouse_root(name: str) -> str:
    try:
        # Preferred - ask Fabric for this lakehouse by name
        lh = notebookutils.lakehouse.get(name)
        # The lookup can return a dict or an object - read its properties either way
        props = lh["properties"] if isinstance(lh, dict) else lh.properties
        # Take its OneLake Tables path and cut off /Tables to leave the lakehouse root
        return props["oneLakeTablesPath"].rstrip("/").rsplit("/Tables", 1)[0]
    except Exception:
        # Fallback if the lookup fails - build the OneLake path from the current workspace name
        ws = notebookutils.runtime.context["currentWorkspaceName"]
        return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{name}.Lakehouse"

# Build the full path to a Delta table - root/Tables/dbo/TableName
def table_path(root, table, schema="dbo"):
    return f"{root}/Tables/{schema}/{table}"

# Root paths of the Bronze, Silver and Gold lakehouses
ROOTS = {"Bronze": lakehouse_root("FP_Bronze"), "Silver": lakehouse_root("FP_Silver"), "Gold": lakehouse_root("FP_Gold")}
# Tables already read in this run, so each is only loaded once
_cache = {}
# Read a table from a layer, reusing the cached copy if it has already been read
def t(layer, table):
    key = (layer, table)
    if key not in _cache:
        # First time only - read the table and keep it in memory
        _cache[key] = spark.read.format("delta").load(table_path(ROOTS[layer], table)).cache()
    return _cache[key]

# One entry per check, saved as the DQ results table at the end
results = []
# Score one check and log the result
def record(layer, table, check, check_type, severity, rows_checked, failed_rows, threshold_pct=0.0, details=""):
    # Failed rows as a percentage of rows checked; 0 if there were no rows
    failed_pct = round(100.0 * failed_rows / rows_checked, 4) if rows_checked else 0.0
    # The check passes if the failure rate is within the allowed threshold - zero by default
    passed = failed_pct <= threshold_pct
    # PASS if within threshold; otherwise FAIL for Critical checks and WARN for the rest
    status = "PASS" if passed else ("FAIL" if severity == "Critical" else "WARN")
    # Add one result row to the in-memory results list
    results.append((run_id, run_ts, layer, table, check, check_type, severity, int(rows_checked), int(failed_rows),
                    float(failed_pct), float(threshold_pct), status, details))

In [ ]:
# ---------- 1. Row count reconciliation: Bronze (mirror) vs Silver ----------
# Full-refresh tables should have the same row count in Bronze and Silver
for tbl in ["Branch", "Product", "LoanApplication", "Loan", "Repayment", "Arrears"]:
    # Count rows in Bronze and in Silver
    b, s = t("Bronze", tbl).count(), t("Silver", tbl).count()
    # Record the difference between the two counts as the failed rows
    record("Silver", tbl, "Row count matches Bronze", "Reconciliation", "Critical", b, abs(b - s),
           details=f"bronze={b}, silver={s}")
# Customer keeps history in Silver, so compare Bronze with the current Silver rows only
b, s = t("Bronze", "Customer").count(), t("Silver", "Customer").filter("IsCurrent").count()
# Record the difference between the two counts as the failed rows
record("Silver", "Customer", "Current customers match Bronze", "Reconciliation", "Critical", b, abs(b - s),
       details=f"bronze={b}, silver_current={s}")

# ---------- 2. Completeness ----------
# Columns that must never be empty, per Silver table
REQUIRED = {
    "Customer": ["CustomerID", "CustomerRef", "LastName", "DateOfBirth", "Postcode", "CreditScore"],
    "LoanApplication": ["ApplicationID", "CustomerID", "ProductID", "ApplicationDate", "Status"],
    "Loan": ["LoanID", "CustomerID", "ProductID", "BranchID", "StartDate", "PrincipalAmount", "InterestRatePct"],
    "Repayment": ["RepaymentID", "LoanID", "DueDate", "DueAmount", "Status"],
}
# Check every required column in each table
for tbl, cols in REQUIRED.items():
    df = t("Silver", tbl)
    for c in cols:
        # Failed rows = rows where this column is NULL
        record("Silver", tbl, f"{c} is not null", "Completeness", "Critical", df.count(), df.filter(F.col(c).isNull()).count())

# ---------- 3. Uniqueness ----------
# Each table's business key must appear only once
for tbl, key in {"Branch": "BranchID", "Product": "ProductID", "LoanApplication": "ApplicationID",
                 "Loan": "LoanID", "Repayment": "RepaymentID", "Arrears": "ArrearsID"}.items():
    df = t("Silver", tbl)
    # Failed rows = total rows minus distinct keys, i.e. the number of duplicates
    record("Silver", tbl, f"{key} is unique", "Uniqueness", "Critical", df.count(), df.count() - df.select(key).distinct().count())
# SCD2 checks on the Silver Customer table
cust = t("Silver", "Customer")
# Count customers that do not have exactly one current row
bad_current = cust.groupBy("CustomerID").agg(F.sum(F.col("IsCurrent").cast("int")).alias("n")).filter("n <> 1").count()
record("Silver", "Customer", "Exactly one current SCD2 row per customer", "Uniqueness", "Critical",
       cust.select("CustomerID").distinct().count(), bad_current)
# Every customer version must have a date range that runs forwards
record("Silver", "Customer", "SCD2 ValidFrom < ValidTo", "Validity", "Critical", cust.count(),
       cust.filter(F.col("ValidFrom") >= F.col("ValidTo")).count())

# ---------- 4. Referential integrity ----------
# Count child rows whose key has no matching parent row (orphans)
def orphans(child, child_col, parent, parent_col):
    # A left anti join keeps only child rows with no match in the parent
    return child.join(parent.select(F.col(parent_col).alias("_k")).distinct(), F.col(child_col) == F.col("_k"), "left_anti").count()

# Current customer rows only - the parent for customer links in Silver
cur_cust = cust.filter("IsCurrent")
# Relationships to check - layer, child table, child column, child data, parent data, parent column
RI = [
    # Silver - every child ID must exist in its parent table
    ("Silver", "LoanApplication", "CustomerID", t("Silver", "LoanApplication"), cur_cust, "CustomerID"),
    ("Silver", "Loan", "CustomerID", t("Silver", "Loan"), cur_cust, "CustomerID"),
    ("Silver", "Loan", "ProductID", t("Silver", "Loan"), t("Silver", "Product"), "ProductID"),
    ("Silver", "Loan", "BranchID", t("Silver", "Loan"), t("Silver", "Branch"), "BranchID"),
    ("Silver", "Repayment", "LoanID", t("Silver", "Repayment"), t("Silver", "Loan"), "LoanID"),
    ("Silver", "Arrears", "LoanID", t("Silver", "Arrears"), t("Silver", "Loan"), "LoanID"),
    # Gold - every fact key must exist in its dimension
    ("Gold", "FactRepayment", "CustomerSK", t("Gold", "FactRepayment"), t("Gold", "DimCustomer"), "CustomerSK"),
    ("Gold", "FactRepayment", "ProductKey", t("Gold", "FactRepayment"), t("Gold", "DimProduct"), "ProductKey"),
    ("Gold", "FactRepayment", "BranchKey", t("Gold", "FactRepayment"), t("Gold", "DimBranch"), "BranchKey"),
    ("Gold", "FactRepayment", "DueDateKey", t("Gold", "FactRepayment"), t("Gold", "DimDate"), "DateKey"),
    ("Gold", "FactLoanSnapshot", "CustomerSK", t("Gold", "FactLoanSnapshot"), t("Gold", "DimCustomer"), "CustomerSK"),
    ("Gold", "FactLoanSnapshot", "SnapshotDateKey", t("Gold", "FactLoanSnapshot"), t("Gold", "DimDate"), "DateKey"),
]
for layer, tbl, col, child, parent, pcol in RI:
    # Failed rows = orphan count
    record(layer, tbl, f"{col} exists in parent", "Referential integrity", "Critical", child.count(), orphans(child, col, parent, pcol))
for tbl in ["FactRepayment", "FactLoanSnapshot"]:
    df = t("Gold", tbl)
    # Warn if any fact rows fell back to the Unknown customer (key -1)
    record("Gold", tbl, "No rows mapped to Unknown customer", "Referential integrity", "Warning", df.count(),
           df.filter("CustomerSK = -1").count())

# ---------- 5. Validity / value ranges ----------
# Short names for the Silver tables used below (customers are current rows only)
c, l, r, a, app = cur_cust, t("Silver", "Loan"), t("Silver", "Repayment"), t("Silver", "Arrears"), t("Silver", "LoanApplication")
# Customer age in years on the as-at date
age = F.months_between(F.lit(AS_AT), F.col("DateOfBirth")) / 12
# Pattern for a valid UK postcode, e.g. WS1 2AB
UK_POSTCODE = r"^[A-Z]{1,2}[0-9][0-9A-Z]? [0-9][A-Z]{2}$"
# Pattern for a well-formed e-mail address
EMAIL = r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$"
VALIDITY = [
    # Each rule - table name, its data, check name, condition that marks a row as failing, severity, allowed failure %
    ("Customer", c, "CreditScore between 0 and 999", ~F.col("CreditScore").between(0, 999), "Critical", 0.0),
    ("Customer", c, "Age between 18 and 100", ~age.between(18, 100), "Critical", 0.0),
    ("Customer", c, "Postcode is valid UK format", ~F.col("Postcode").rlike(UK_POSTCODE), "Warning", 0.5),
    ("Customer", c, "Email is valid format", ~F.col("Email").rlike(EMAIL), "Warning", 0.5),
    ("Customer", c, "AnnualIncome >= 0", F.col("AnnualIncome") < 0, "Critical", 0.0),
    ("Loan", l, "InterestRatePct between 0 and 30", ~F.col("InterestRatePct").between(0, 30), "Critical", 0.0),
    ("Loan", l, "PrincipalAmount > 0", F.col("PrincipalAmount") <= 0, "Critical", 0.0),
    ("Loan", l, "OutstandingBalance <= PrincipalAmount", F.col("OutstandingBalance") > F.col("PrincipalAmount"), "Critical", 0.0),
    ("Loan", l, "Status in allowed list", ~F.col("Status").isin("Active", "Arrears", "Default", "Closed"), "Critical", 0.0),
    ("Repayment", r, "PaidAmount between 0 and DueAmount", (F.col("PaidAmount") < 0) | (F.col("PaidAmount") > F.col("DueAmount")), "Critical", 0.0),
    ("Repayment", r, "Missed instalments have no PaidDate", (F.col("Status") == "Missed") & F.col("PaidDate").isNotNull(), "Critical", 0.0),
    ("Repayment", r, "DueDate not after as-at date", F.col("DueDate") > F.lit(AS_AT), "Critical", 0.0),
    ("Arrears", a, "DaysPastDue >= 1", F.col("DaysPastDue") < 1, "Critical", 0.0),
    ("LoanApplication", app, "ApplicationDate not in the future", F.col("ApplicationDate") > F.lit(AS_AT), "Critical", 0.0),
    ("LoanApplication", app, "Declined applications have a reason",
        (F.col("Status") == "Declined") & F.col("DeclineReason").isNull(), "Warning", 0.0),
]
# Run each validity rule
for tbl, df, name, fail_cond, sev, thr in VALIDITY:
    # Failed rows = rows that meet the failing condition
    record("Silver", tbl, name, "Validity", sev, df.count(), df.filter(fail_cond).count(), thr)

# ---------- 6. Reasonableness (business sanity) ----------
# Loan snapshot rows for the as-at date
latest = t("Gold", "FactLoanSnapshot").filter(F.col("SnapshotDate") == F.lit(AS_AT))
# Count open loans, and open loans in arrears
open_loans = latest.filter("IsOpen = 1").count()
in_arrears = latest.filter("IsOpen = 1 AND IsInArrears = 1").count()
# Arrears rate as a percentage of open loans
rate = 100.0 * in_arrears / open_loans if open_loans else 0
# Warn if the rate falls outside 0.5% to 15% - a single pass/fail result for the whole portfolio
record("Gold", "FactLoanSnapshot", "Arrears rate within 0.5%–15%", "Reasonableness", "Warning", 1,
       0 if 0.5 <= rate <= 15 else 1, details=f"arrears rate = {rate:.2f}% ({in_arrears}/{open_loans})")

In [ ]:
# ---------- Persist results to FP_Gold.dbo.DQ_Results and show the summary ----------
# Column names and types for the results table - one row per check
schema = StructType([
    StructField("RunId", StringType()), StructField("RunTimestamp", TimestampType()),
    StructField("Layer", StringType()), StructField("TableName", StringType()),
    StructField("CheckName", StringType()), StructField("CheckType", StringType()),
    StructField("Severity", StringType()), StructField("RowsChecked", LongType()),
    StructField("FailedRows", LongType()), StructField("FailedPct", DoubleType()),
    StructField("ThresholdPct", DoubleType()), StructField("Status", StringType()),
    StructField("Details", StringType())])
# Turn the results list into a Spark table
dq = spark.createDataFrame(results, schema)
# Append to DQ_Results in Gold, so every run's results are kept as history
dq.write.format("delta").mode("append").option("mergeSchema", "true").save(table_path(ROOTS["Gold"], "DQ_Results"))

# Count checks by status and print a one-line summary
summary = {s: dq.filter(F.col("Status") == s).count() for s in ["PASS", "WARN", "FAIL"]}
print(f"Checks run: {dq.count()}  |  PASS {summary['PASS']}  WARN {summary['WARN']}  FAIL {summary['FAIL']}")
# Show every result, failures first, then warnings, then passes
display(dq.select("Layer", "TableName", "CheckName", "CheckType", "Severity", "RowsChecked", "FailedRows", "Status", "Details")
          .orderBy(F.expr("CASE Status WHEN 'FAIL' THEN 0 WHEN 'WARN' THEN 1 ELSE 2 END"), "Layer", "TableName"))

In [ ]:
# Stop here if fail_on_critical is on and any Critical check failed
if fail_on_critical and summary["FAIL"] > 0:
    # List the failed checks by name and table, then raise an error so the pipeline takes its failure path
    failed = [r[4] + " (" + r[3] + ")" for r in results if r[11] == "FAIL"]
    raise Exception(f"Data quality FAILED – {summary['FAIL']} critical check(s): {failed}")

# Otherwise end the notebook and hand a JSON summary back to the pipeline - run ID, status and check counts
notebookutils.notebook.exit(json.dumps({"notebook": "FP_NB_DataQuality", "run_id": run_id, "status": "Succeeded",
                                        "checks": len(results), **summary}))